In [1]:
import numpy as np
import pickle
import matplotlib.pyplot as plt
import pandas as pd
import datetime
from IQAEBinom_OnlyFinalRound_EndCIReachTol import IQAEBinom_OnlyFinalRound_EndCIReachTol

In [2]:
nShotUnit = 1
epsilon = 0.001
alpha = 0.05

In [3]:
pTrue = 0.2505
thetaTrue = np.arcsin(np.sqrt(pTrue))

In [5]:
# 次のroundに行く場合はNaNとする
nEstim = 10000
minRatio = 2
nGrover = 300
angleFracs = [0.4878048780487805, 0.5121951219512195] # (2*nGrov + 1) * theta / pi の小数部分の意

r = int((2 * nGrover + 1) * thetaTrue / np.pi)
results = []

for angleFrac in angleFracs:
    thetaTemp = (r + angleFrac) * np.pi / (2 * nGrover + 1)
    p = np.sin(thetaTemp) ** 2
    resultsTemp = []
    for i in range(nEstim):
        result = IQAEBinom_OnlyFinalRound_EndCIReachTol(p, nGrover, epsilon, alpha, nShotUnit, minRatio=2)
        resultsTemp.append(result)
    results.append(resultsTemp)

In [6]:
results[0][0]

{'Estimate': 0.2515104994630764,
 'TotalOracleCalls': 8414,
 'nShots': 14,
 'n1s': 14,
 'thetaIntervals': [0.5242109212980371, 0.5253412008082766],
 'ampSqIntervals': [0.2505303209555109, 0.2515104994630764]}

In [7]:
results[1][0]

{'Estimate': nan,
 'TotalOracleCalls': 5409,
 'nShots': 9,
 'n1s': 9,
 'thetaIntervals': [0.5253412008082766, 0.526635157764672],
 'ampSqIntervals': [0.2515104994630764, 0.2526341777233653]}

angle frac=0.4878048780487805 と 0.5121951219512195で、1が出続けた場合の挙動が異なる！

前者は精度εが達成されて終了するが、後者は次のroundに行く！

In [15]:
# 1が出続けるとして、theta CI、ならびに、next roundの初期CIとして使われる拡大されたCIを出力
k = 2 * nGrover + 1
kmax = np.pi / 4 / epsilon
alphaRound = 2 * alpha / 3 * k / kmax

nShotRound = 0
n1Round = 0

for angleFrac in angleFracs:
    rTemp = int((2 * nGrover + 1) * thetaTrue / np.pi)
    theta = (rTemp + angleFrac) * np.pi / (2 * nGrover + 1)
    r = int(k * theta / (0.5 * np.pi))
    print(angleFrac)
    for nShot in range(1, 100): # 100:iteration limit
        p1Width = np.sqrt(0.5 / nShot * np.log(2 / alphaRound))
        p1Max = 1
        p1Min = max(1 - p1Width, 0)

        if r % 2 == 0:
            gammaMin = np.arcsin(np.sqrt(p1Min))
            gammaMax = np.arcsin(np.sqrt(p1Max))
            gammaML = np.arcsin(np.sqrt(1))
        else:
            gammaMin = 0.5 * np.pi - np.arcsin(np.sqrt(p1Max))
            gammaMax = 0.5 * np.pi - np.arcsin(np.sqrt(p1Min))
            gammaML = 0.5 * np.pi - np.arcsin(np.sqrt(1))

        theta_u = (r * 0.5 * np.pi + gammaMax) / k
        theta_l = (r * 0.5 * np.pi + gammaMin) / k
        thetaInterval = [theta_l, theta_u]
        thetaML = (r * 0.5 * np.pi + gammaML) / k

        kNextMin = 2 * k
        R_u = np.ceil(kNextMin * theta_u / (0.5 * np.pi)) - 1
        R_l = int(kNextMin * theta_l / (0.5 * np.pi))

        print(nShot, theta_l, thetaML, theta_u, R_l, R_u)

        ampSq_u = np.sin(theta_u) ** 2
        ampSq_l = np.sin(theta_l) ** 2
        ampSqInterval = [ampSq_l, ampSq_u]
        ampSqML = np.sin(thetaML) ** 2
        ampSqWidth = max(ampSq_u - ampSqML, ampSqML - ampSq_l)

        if ampSqWidth <= epsilon:
            break

0.4878048780487805
1 0.52272756299331 0.5253412008082766 0.5253412008082766 400 402.0
2 0.52272756299331 0.5253412008082766 0.5253412008082766 400 402.0
3 0.5233831257212792 0.5253412008082766 0.5253412008082766 400 402.0
4 0.5236208852557889 0.5253412008082766 0.5253412008082766 400 402.0
5 0.5237626020872611 0.5253412008082766 0.5253412008082766 400 402.0
6 0.5238619256887176 0.5253412008082766 0.5253412008082766 400 402.0
7 0.5239373533894329 0.5253412008082766 0.5253412008082766 400 402.0
8 0.523997545448445 0.5253412008082766 0.5253412008082766 400 402.0
9 0.5240472438518812 0.5253412008082766 0.5253412008082766 401 402.0
10 0.5240893160072777 0.5253412008082766 0.5253412008082766 401 402.0
11 0.5241256208946493 0.5253412008082766 0.5253412008082766 401 402.0
12 0.5241574280765597 0.5253412008082766 0.5253412008082766 401 402.0
13 0.5241856404317118 0.5253412008082766 0.5253412008082766 401 402.0
14 0.5242109212980371 0.5253412008082766 0.5253412008082766 401 402.0
0.5121951219512

FindNextKがangle fracに対して対称ではないから、biasが対称ではない！